# ETL Pipeline: Livestock Intelligence - EXTRACT Phase
**Laporan Praktikum Teknologi Perekayasaan Data - Kelompok 1 (3SI1)**

Pipeline Extract mengambil data mentah (*raw*) dari 3 sumber heterogen:
1. **BPS** - API Scraping + Dummy Generator → PostgreSQL (`bps_db`) → `staging_db`
2. **iSIKHNAS** - MySQL (`isikhnas_db`) → PostgreSQL (`staging_db`)
3. **PIHPS** - File Excel (`final_data.xlsx`) → PostgreSQL (`staging_db`)

> **Catatan:** Tahap ini murni Extract. Tidak ada pembersihan, imputasi, atau transformasi apapun.

In [4]:
import pandas as pd
import numpy as np
import random
import requests
import time
from tqdm import tqdm
from sqlalchemy import create_engine
import warnings
warnings.filterwarnings('ignore')

# =================== KONFIGURASI DATABASE ===================
PG_USER = 'postgres'
PG_PASS = 'postgres'       # Sesuaikan
PG_HOST = 'localhost'
PG_PORT = '5432'

MYSQL_USER = 'root'
MYSQL_PASS = ''             # Sesuaikan (kosong jika default XAMPP)
MYSQL_HOST = 'localhost'
MYSQL_PORT = '3306'

# Engine PostgreSQL
engine_bps    = create_engine(f'postgresql://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/bps_db')
engine_staging = create_engine(f'postgresql://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/staging_db')

# Engine MySQL
engine_mysql  = create_engine(f'mysql+pymysql://{MYSQL_USER}:{MYSQL_PASS}@{MYSQL_HOST}:{MYSQL_PORT}/isikhnas_db')

print("Koneksi database dikonfigurasi.")

Koneksi database dikonfigurasi.


## 1. Extract Data BPS (API Scraping + Dummy)

### Variabel dari Scraping API:
| Variabel | Satuan | Sumber API |
|---|---|---|
| `provinsi` | - | SIMDASI |
| `tahun` | - | Parameter |
| `jumlah_penduduk` | jiwa | SIMDASI |
| `populasi_sapi` | ekor | SIMDASI |
| `populasi_ayam` | ekor | SIMDASI |
| `produksi_daging_ayam` | kg | SIMDASI |

### Variabel dari Dummy Generator:
`produksi_daging_sapi`, `konsumsi_daging_sapi`, `konsumsi_daging_ayam`, `permintaan_daging_sapi`, `permintaan_daging_ayam`, `jumlah_ternak_sapi_potong`, `jumlah_ternak_ayam_potong`, `harga_baseline_sapi`, `harga_baseline_ayam`, `growth_populasi`, `status_supply`

In [1]:
# =========================================================
# KONFIGURASI API BPS & FUNGSI INTI
# Sumber referensi: scraping_bps.ipynb (sudah teruji)
# =========================================================
API_KEY = "d328d5f200379241367024848106698e"
YEARS   = [2020, 2021, 2022, 2023, 2024, 2025]

# --- Komoditas yang di-scrape (4 variabel) ---
COMMODITIES_TO_SCRAPE = [
    {
        "name": "Jumlah Penduduk",
        "table_ids": ["WVRlTTcySlZDa3lUcFp6czNwbHl4QT09"],
        "keywords": ["jumlah", "penduduk"],
        "column": "jumlah_penduduk"
    },
    {
        "name": "Populasi Ternak - Sapi Potong",
        "table_ids": ["S2ViU1dwVTlpSXRwU1MvendHN05Cdz09"],
        "keywords": ["populasi", "sapi", "potong"],
        "column": "populasi_sapi_potong"
    },
    {
        "name": "Populasi Unggas - Ayam Pedaging",
        "table_ids": ["ckJyVXRMT05MWTNpaW9mdnVseFk0Zz09"],
        "keywords": ["populasi", "ayam", "pedaging"],
        "column": "populasi_ayam_pedaging"
    },
    {
        "name": "Produksi Daging Ayam Pedaging",
        "table_ids": ["dWhmNFl6WXYyR093R2NjTGM3NG9idz09"],
        "keywords": ["produksi", "daging", "ayam", "pedaging"],
        "column": "produksi_daging_ayam_pedaging"
    }
]

# =================== FUNGSI INTI ===================
def get_bps_data(year, table_id, kode_wilayah="0000000"):
    url = (
        f"https://webapi.bps.go.id/v1/api/interoperabilitas/datasource/simdasi/"
        f"id/25/tahun/{year}/id_tabel/{table_id}/"
        f"wilayah/{kode_wilayah}/key/{API_KEY}"
    )
    try:
        resp = requests.get(url, timeout=30)
        return resp.json()
    except:
        return None

def normalize_prov(nama):
    return (
        nama.lower()
        .replace(".", "")
        .replace("provinsi", "")
        .replace("kep.", "kepulauan")
        .strip()
    )

def get_target_var_id(main_data, keywords):
    for vid, vinfo in main_data.get("kolom", {}).items():
        nama = str(vinfo.get("nama_variabel", "")).lower()
        if all(k in nama for k in keywords):
            return vid
    return None

def parse_province_data(raw, year, prov_name, keywords, col_name):
    if not raw or raw.get("status") != "OK":
        return pd.DataFrame()
    try:
        md = raw.get("data", [{}, {}])[1]
        vid = get_target_var_id(md, keywords)
        if not vid:
            return pd.DataFrame()
        for item in md.get("data", []):
            if item.get("label", "").lower() == prov_name.lower():
                val = item.get("variables", {}).get(vid, {}).get("value_raw")
                return pd.DataFrame([{
                    "tahun": year, "provinsi": prov_name, col_name: val
                }])
        return pd.DataFrame()
    except:
        return pd.DataFrame()

print("Fungsi API BPS siap.")

Fungsi API BPS siap.


### Pre-Flight Check: Ketersediaan Data API
> **Cell ini hanya untuk verifikasi.** Hapus setelah konfirmasi data tersedia.

In [6]:
# =========================================================
# PRE-FLIGHT CHECK - Cek ketersediaan data API BPS
# Cell ini bisa dihapus setelah verifikasi selesai
# =========================================================
print("Menguji ketersediaan data API BPS...\n")

hasil_cek = []
for comm in COMMODITIES_TO_SCRAPE:
    row = {"Komoditas": comm["name"]}
    for y in YEARS:
        status = "X"
        for tbl in comm["table_ids"]:
            temp = get_bps_data(y, tbl, "0000000")
            if temp and temp.get("status") == "OK":
                md_check = temp.get("data", [{}, {}])[1]
                if get_target_var_id(md_check, comm["keywords"]):
                    status = "OK"
                    break
        row[str(y)] = status
    hasil_cek.append(row)

df_cek = pd.DataFrame(hasil_cek)
print(df_cek.to_string(index=False))
print("\nJika semua OK, lanjut ke cell scraping berikutnya.")

Menguji ketersediaan data API BPS...

                      Komoditas 2020 2021 2022 2023 2024 2025
                Jumlah Penduduk   OK   OK   OK   OK   OK   OK
  Populasi Ternak - Sapi Potong   OK   OK   OK   OK   OK   OK
Populasi Unggas - Ayam Pedaging   OK   OK   OK   OK   OK    X
  Produksi Daging Ayam Pedaging   OK   OK   OK   OK   OK   OK

Jika semua OK, lanjut ke cell scraping berikutnya.


### 1.1 Scraping API BPS
Proses ini memakan waktu ~15-25 menit karena harus meng-hit API per provinsi per tahun per komoditas.

In [ ]:
# =========================================================
# SCRAPING UTAMA - Disesuaikan dengan scraping_bps.ipynb
# 4 Komoditas x 6 Tahun x ~34 Provinsi
# =========================================================
print("Memulai proses scraping API BPS...\n")

final_merged_df = pd.DataFrame()

for idx_comm, comm in enumerate(COMMODITIES_TO_SCRAPE):
    name = comm["name"]
    keywords = comm["keywords"]
    col_name = comm["column"]

    if idx_comm > 0:
        print(f"\n--- Jeda 10 detik sebelum komoditas berikutnya... ---")
        time.sleep(10)

    print(f"\n--- Mengambil data: {name} ---")
    comm_dfs = []

    for year in YEARS:
        print(f"[{year}] Mencari data untuk {name}...")

        active_tbl = None
        raw_nasional = None

        # Cari table ID yang valid untuk tahun ini
        for tbl in comm["table_ids"]:
            temp = get_bps_data(year, tbl, "0000000")
            if temp and temp.get("status") == "OK":
                md_check = temp.get("data", [{}, {}])[1]
                if get_target_var_id(md_check, keywords):
                    active_tbl = tbl
                    raw_nasional = temp
                    break

        if not active_tbl or not raw_nasional:
            print(f"  X Data tidak tersedia untuk {name} tahun {year}, skip.")
            continue

        print(f"  OK Tabel valid ditemukan ({active_tbl}). Mulai drill-down...")

        # Ambil daftar provinsi dari data nasional
        data_nasional = raw_nasional.get("data", [{}, {}])[1]
        list_prov = [
            {"nama": it.get("label", ""), "kode": it.get("kode_wilayah", "")}
            for it in data_nasional.get("data", [])
            if it.get("label", "").lower() != "indonesia" and it.get("kode_wilayah", "")
        ]

        year_dfs = []
        for prov in tqdm(list_prov, desc=f"Scraping {year} {name}"):
            time.sleep(0.3)
            raw_prov = get_bps_data(year, active_tbl, prov["kode"])
            df_prov = parse_province_data(raw_prov, year, prov["nama"], keywords, col_name)
            if not df_prov.empty:
                year_dfs.append(df_prov)

        if year_dfs:
            comm_dfs.append(pd.concat(year_dfs, ignore_index=True))

    if comm_dfs:
        full_df = pd.concat(comm_dfs, ignore_index=True)
        if final_merged_df.empty:
            final_merged_df = full_df
        else:
            final_merged_df = pd.merge(
                final_merged_df, full_df,
                on=['tahun', 'provinsi'], how='outer'
            )

# Rename kolom sesuai spesifikasi
df_scraped = final_merged_df.rename(columns={
    'populasi_sapi_potong': 'populasi_sapi',
    'populasi_ayam_pedaging': 'populasi_ayam',
    'produksi_daging_ayam_pedaging': 'produksi_daging_ayam'
})

# Parsing angka (cleaning format BPS)
for col in ['jumlah_penduduk', 'populasi_sapi', 'populasi_ayam', 'produksi_daging_ayam']:
    if col in df_scraped.columns:
        df_scraped[col] = df_scraped[col].astype(str)
        df_scraped[col] = (
            df_scraped[col]
            .str.replace('.', '', regex=False)
            .str.replace(',', '.', regex=False)
        )
        df_scraped[col] = pd.to_numeric(df_scraped[col], errors='coerce')

print(f"\nScraping selesai! {len(df_scraped)} baris")
print(df_scraped.head())
print(df_scraped.isna().sum())

### 1.2 Generate Data Dummy & Merge dengan Hasil Scraping
Dummy data di-generate untuk **semua** variabel. Lalu, nilai dari scraping menimpa (*overwrite*) nilai dummy pada kolom yang bersesuaian menggunakan key `provinsi` + `tahun`.

In [ ]:
# =========================================================
# GENERATE DATA DUMMY & MERGE
# =========================================================
PROVINSI_LIST = [
    "Aceh","Sumatera Utara","Sumatera Barat","Riau","Kepulauan Riau","Jambi",
    "Sumatera Selatan","Kepulauan Bangka Belitung","Bengkulu","Lampung","DKI Jakarta",
    "Jawa Barat","Jawa Tengah","DI Yogyakarta","Jawa Timur","Banten","Bali",
    "Nusa Tenggara Barat","Nusa Tenggara Timur","Kalimantan Barat",
    "Kalimantan Tengah","Kalimantan Selatan","Kalimantan Timur",
    "Kalimantan Utara","Sulawesi Utara","Sulawesi Tengah","Sulawesi Selatan",
    "Sulawesi Tenggara","Gorontalo","Sulawesi Barat","Maluku","Maluku Utara",
    "Papua","Papua Barat","Papua Selatan","Papua Tengah",
    "Papua Pegunungan","Papua Barat Daya"
]

def generate_bps_dummy():
    random.seed(42)  # Reproducibility
    data = []
    for prov in PROVINSI_LIST:
        base_pop = random.randint(500000, 50000000)
        for tahun in YEARS:
            growth = round(random.uniform(-0.02, 0.04), 4)
            jml_penduduk = int(base_pop * (1 + growth))
            pop_sapi = random.randint(10000, 500000)
            pop_ayam = random.randint(50000, 5000000)
            potong_sapi = int(pop_sapi * random.uniform(0.4, 0.7))
            potong_ayam = int(pop_ayam * random.uniform(0.5, 0.8))
            prod_sapi = round(potong_sapi * random.uniform(0.2, 0.3), 2)
            prod_ayam = round(potong_ayam * random.uniform(0.1, 0.2), 2)
            kons_sapi = round(random.uniform(1.5, 3.5), 2)
            kons_ayam = round(random.uniform(8, 15), 2)
            perm_sapi = round(jml_penduduk * kons_sapi / 1000, 2)
            perm_ayam = round(jml_penduduk * kons_ayam / 1000, 2)
            harga_sapi = random.randint(90000, 130000)
            harga_ayam = random.randint(20000, 40000)
            
            # Status supply
            supply_ratio = pop_sapi / max(perm_sapi, 1)
            if supply_ratio > 1.2: status = "Surplus"
            elif supply_ratio > 0.8: status = "Seimbang"
            else: status = "Defisit"
            
            row = {
                "provinsi": prov, "tahun": tahun,
                "jumlah_penduduk": jml_penduduk,
                "populasi_sapi": pop_sapi, "populasi_ayam": pop_ayam,
                "produksi_daging_sapi": prod_sapi, "produksi_daging_ayam": prod_ayam,
                "konsumsi_daging_sapi": kons_sapi, "konsumsi_daging_ayam": kons_ayam,
                "permintaan_daging_sapi": perm_sapi, "permintaan_daging_ayam": perm_ayam,
                "jumlah_ternak_sapi_potong": potong_sapi,
                "jumlah_ternak_ayam_potong": potong_ayam,
                "harga_baseline_sapi": harga_sapi, "harga_baseline_ayam": harga_ayam,
                "growth_populasi": growth, "status_supply": status
            }
            # Inject missing value (realistis ~5%)
            if random.random() < 0.05: row["produksi_daging_sapi"] = None
            if random.random() < 0.05: row["permintaan_daging_ayam"] = None
            if random.random() < 0.05: row["harga_baseline_sapi"] = None
            if random.random() < 0.03: row["jumlah_ternak_sapi_potong"] = None
            data.append(row)
    return pd.DataFrame(data)

# Generate dummy
df_dummy = generate_bps_dummy()
print(f"Dummy generated: {df_dummy.shape}")

# FIX: Konversi kolom numerik ke float agar kompatibel saat update
numeric_cols = [
    'jumlah_penduduk', 'populasi_sapi', 'populasi_ayam',
    'produksi_daging_sapi', 'produksi_daging_ayam',
    'konsumsi_daging_sapi', 'konsumsi_daging_ayam',
    'permintaan_daging_sapi', 'permintaan_daging_ayam',
    'jumlah_ternak_sapi_potong', 'jumlah_ternak_ayam_potong',
    'harga_baseline_sapi', 'harga_baseline_ayam'
]
for col in numeric_cols:
    if col in df_dummy.columns:
        df_dummy[col] = df_dummy[col].astype(float)

# Merge: scraped data menimpa dummy
df_dummy_indexed = df_dummy.set_index(['provinsi', 'tahun'])
df_scraped_indexed = df_scraped.set_index(['provinsi', 'tahun'])

# Pastikan kolom scraped juga float
for col in df_scraped_indexed.columns:
    if col in numeric_cols:
        df_scraped_indexed[col] = pd.to_numeric(df_scraped_indexed[col], errors='coerce').astype(float)

# Update: nilai scraping overwrite nilai dummy
df_dummy_indexed.update(df_scraped_indexed)
df_bps_full = df_dummy_indexed.reset_index()

print(f"Merged BPS data: {df_bps_full.shape}")
print(f"Provinsi: {df_bps_full['provinsi'].nunique()}")
print(f"\nNull count:")
print(df_bps_full.isnull().sum())
df_bps_full.head()

### 1.3 Normalisasi ke 4 Tabel Relasional
Data flat di-UNPIVOT menjadi dimensi komoditas (Sapi & Ayam) sesuai skema:
- `ref_wilayah` (PK: provinsi)
- `ref_komoditas` (PK: id_komoditas)
- `tr_demografi_tahunan` (PK: provinsi, tahun)
- `tr_statistik_peternakan` (PK: provinsi, tahun, id_komoditas)

In [ ]:
# =========================================================
# NORMALISASI KE 4 TABEL
# =========================================================

# 1. ref_wilayah
df_ref_wilayah = pd.DataFrame({
    'provinsi': sorted(df_bps_full['provinsi'].unique())
})

# 2. ref_komoditas
df_ref_komoditas = pd.DataFrame({
    'id_komoditas': [1, 2],
    'nama_komoditas': ['Sapi', 'Ayam'],
    'satuan_berat': ['Ton', 'Kg']
})

# 3. tr_demografi_tahunan
df_demografi = df_bps_full[['provinsi','tahun','jumlah_penduduk','growth_populasi','status_supply']].drop_duplicates()

# 4. tr_statistik_peternakan (UNPIVOT Sapi & Ayam)
rows_stat = []
for _, r in df_bps_full.iterrows():
    # Baris Sapi (id_komoditas = 1)
    rows_stat.append({
        'provinsi': r['provinsi'], 'tahun': r['tahun'], 'id_komoditas': 1,
        'populasi': r.get('populasi_sapi'),
        'produksi': r.get('produksi_daging_sapi'),
        'konsumsi_per_kapita': r.get('konsumsi_daging_sapi'),
        'permintaan_ekor': r.get('permintaan_daging_sapi'),
        'jumlah_dipotong': r.get('jumlah_ternak_sapi_potong'),
        'harga_baseline': r.get('harga_baseline_sapi')
    })
    # Baris Ayam (id_komoditas = 2)
    rows_stat.append({
        'provinsi': r['provinsi'], 'tahun': r['tahun'], 'id_komoditas': 2,
        'populasi': r.get('populasi_ayam'),
        'produksi': r.get('produksi_daging_ayam'),
        'konsumsi_per_kapita': r.get('konsumsi_daging_ayam'),
        'permintaan_ekor': r.get('permintaan_daging_ayam'),
        'jumlah_dipotong': r.get('jumlah_ternak_ayam_potong'),
        'harga_baseline': r.get('harga_baseline_ayam')
    })

df_statistik = pd.DataFrame(rows_stat)

print("=== HASIL NORMALISASI ===")
print(f"ref_wilayah         : {df_ref_wilayah.shape}")
print(f"ref_komoditas       : {df_ref_komoditas.shape}")
print(f"tr_demografi_tahunan: {df_demografi.shape}")
print(f"tr_statistik_peternakan: {df_statistik.shape}")
print()
display(df_ref_wilayah.head())
display(df_ref_komoditas)
display(df_demografi.head())
display(df_statistik.head())

### 1.4 Injeksi ke PostgreSQL
1. Push 4 tabel ke `bps_db` (database sumber BPS)
2. Push 4 tabel yang sama ke `staging_db` (area staging gabungan)

In [15]:
# =========================================================
# LOAD KE POSTGRESQL (bps_db + staging_db)
# =========================================================

# --- Push ke bps_db ---
try:
    df_ref_wilayah.to_sql('ref_wilayah', engine_bps, if_exists='append', index=False)
    df_ref_komoditas.to_sql('ref_komoditas', engine_bps, if_exists='append', index=False)
    df_demografi.to_sql('tr_demografi_tahunan', engine_bps, if_exists='append', index=False)
    df_statistik.to_sql('tr_statistik_peternakan', engine_bps, if_exists='append', index=False)
    print("[bps_db] 4 tabel berhasil dimuat.")
except Exception as e:
    print(f"[bps_db] Error: {e}")

# --- Push ke staging_db ---
try:
    df_ref_wilayah.to_sql('bps_ref_wilayah', engine_staging, if_exists='append', index=False)
    df_ref_komoditas.to_sql('bps_ref_komoditas', engine_staging, if_exists='append', index=False)
    df_demografi.to_sql('bps_tr_demografi_tahunan', engine_staging, if_exists='append', index=False)
    df_statistik.to_sql('bps_tr_statistik_peternakan', engine_staging, if_exists='append', index=False)
    print("[staging_db] 4 tabel BPS berhasil dimuat.")
except Exception as e:
    print(f"[staging_db] Error: {e}")

[bps_db] Error: name 'df_ref_wilayah' is not defined
[staging_db] Error: name 'df_ref_wilayah' is not defined


## 2. Extract Data iSIKHNAS (MySQL → PostgreSQL)
Data iSIKHNAS sudah tersedia di MySQL (`isikhnas_db`) melalui phpMyAdmin.
Tabel yang diekstrak: `ref_hewan`, `ref_wilayah`, `tr_mutasi`, `tr_laporan_sakit`, `tr_hasil_lab`, `tr_rph`.

In [ ]:
# =========================================================
# EXTRACT iSIKHNAS: MySQL → staging_db (PostgreSQL)
# =========================================================
tabel_isikhnas = ['ref_hewan', 'ref_wilayah', 'tr_mutasi', 'tr_laporan_sakit', 'tr_hasil_lab', 'tr_rph']

try:
    for tbl in tabel_isikhnas:
        df_tbl = pd.read_sql(f"SELECT * FROM {tbl}", engine_mysql)
        df_tbl.to_sql(f'isikhnas_{tbl}', engine_staging, if_exists='append', index=False)
        print(f"  [staging_db] isikhnas_{tbl}: {len(df_tbl)} baris dimuat.")
    print("\n[iSIKHNAS] Semua tabel berhasil diekstrak ke staging_db.")
except Exception as e:
    print(f"[iSIKHNAS] Error: {e}")

## 3. Extract Data PIHPS (Excel → PostgreSQL)
Data harga pasar harian dari file `final_data.xlsx`.
Variabel: Provinsi, Kota, Nama_komoditas, Jenis_pasar, Harga, Waktu.

In [ ]:
# =========================================================
# EXTRACT PIHPS: Excel → staging_db (PostgreSQL)
# =========================================================
import os

# Path relatif dari folder CODE
file_pihps = os.path.join(os.path.dirname(os.path.abspath('.')), 'DATA', 'PIHPS', 'final_data.xlsx')

# Fallback: coba beberapa lokasi
possible_paths = [
    file_pihps,
    '../DATA/PIHPS/final_data.xlsx',
    '../DATA/final_data.xlsx',
    'final_data.xlsx'
]

df_pihps = None
for path in possible_paths:
    if os.path.exists(path):
        df_pihps = pd.read_excel(path)
        print(f"File ditemukan: {path}")
        break

if df_pihps is not None:
    try:
        df_pihps.to_sql('pihps_harga_harian', engine_staging, if_exists='append', index=False)
        print(f"[staging_db] pihps_harga_harian: {len(df_pihps)} baris dimuat.")
    except Exception as e:
        print(f"[PIHPS] Error saat load ke DB: {e}")
    display(df_pihps.head())
else:
    print("File final_data.xlsx tidak ditemukan. Periksa lokasi file.")

## Ringkasan Proses Extract
Verifikasi akhir: cek jumlah tabel dan baris yang sudah masuk ke `staging_db`.

In [ ]:
# =========================================================
# RINGKASAN & VALIDASI AKHIR
# =========================================================
from sqlalchemy import inspect

print("=" * 60)
print("RINGKASAN PROSES EXTRACT")
print("=" * 60)

try:
    inspector = inspect(engine_staging)
    tabel_staging = inspector.get_table_names()
    print(f"\nTotal tabel di staging_db: {len(tabel_staging)}")
    print("-" * 40)
    for tbl in sorted(tabel_staging):
        count = pd.read_sql(f"SELECT COUNT(*) as n FROM {tbl}", engine_staging).iloc[0, 0]
        print(f"  {tbl:40s} : {count:>8,} baris")
    print("-" * 40)
    print("\nExtract phase SELESAI.")
except Exception as e:
    print(f"Error validasi: {e}")